In [ ]:
import os

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from torch.nn.functional import softmax, normalize

from pass_pclr.defines import ECHONEXT_TARGETS, ECHONEXT_COMPOSITE_TARGET, PTBXL_CAT1_TARGETS
from pass_pclr.datasets import get_ptbxl_labels

_ = ECHONEXT_TARGETS.pop(ECHONEXT_COMPOSITE_TARGET, None)

In [ ]:
import math

def best_square_subplot_layout(n, fig_width=16, fig_height=9):
    """
    Determine rows, cols, and subplot size (in inches) for n square subplots
    that maximize subplot size within a fixed figure size.
    """
    best = {
        "rows": None,
        "cols": None,
        "subplot_size": 0.0
    }

    # rows can range from 1..n
    for rows in range(1, n + 1):
        cols = math.ceil(n / rows)

        # max square size constrained by width and height
        size_w = fig_width / cols
        size_h = fig_height / rows
        square_size = min(size_w, size_h)

        if square_size > best["subplot_size"]:
            best.update(
                rows=rows,
                cols=cols,
                subplot_size=square_size
            )

    return best["rows"], best["cols"]

## EchoNext

In [ ]:
def get_echonext_labels(meta_path: str) -> tuple[np.ndarray, pd.DataFrame]:
    df = pd.read_csv(meta_path)
    df = df.rename(columns={v: k for k, v in ECHONEXT_TARGETS.items()})
    df = df.loc[df["split"] == "train"].reset_index(drop=True)
    labels = df[list(ECHONEXT_TARGETS)].to_numpy()
    return labels, df

def get_prototypes(ckpt_path: str) -> tuple[np.ndarray, pd.DataFrame]:
    sd = torch.load(os.path.join(ckpt_path, "proj.ckpt"))["state_dict"]
    prototypes = sd["model.encoder.prototypes"]
    meta = pd.read_csv(os.path.join(ckpt_path, "projection_metadata.csv"))
    prototypes = normalize(prototypes, 2)
    return prototypes.numpy(), meta

def get_embeddings(embs_path: str) -> np.ndarray:
    return np.load(os.path.join(embs_path, "train_embeds.npy"))

In [ ]:
def scatter(prototypes, labels, prot_labels):
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(prototypes)
    rows, cols = best_square_subplot_layout(len(labels) + 1)
    fig, axs = plt.subplots(rows, cols, figsize=(16, 9))
    flat_axs = axs.flatten()
    for i, label_name in enumerate(labels):
        ax = flat_axs[i]
        ax.scatter(coords[:, 0], coords[:, 1], c=prot_labels[:, i])
        ax.set_title(label_name)
    return axs

### EchoNext PIP

In [ ]:
pip_meta_path = "/opt/gpudata/ecg/echonext/EchoNext_metadata_100k.csv"
pip_ckpt_path = "../outputs/runs/pass-pretrain-heedb/project-prototypes/latest"
pip_embs_path = "../outputs/runs/pass-heedb-pip-logreg/compute-embeddings/latest"

In [ ]:
labels, labels_meta = get_echonext_labels(pip_meta_path) # (N, L)
pip_prototypes, pip_prot_meta = get_prototypes(pip_ckpt_path)
pip_embeddings = get_embeddings(pip_embs_path) # these are already cosine similarity scores

In [ ]:
# use the odds ratios of the trained logistic regression to find most relevant prototypes for each label?
with np.load("../outputs/runs/pass-heedb-pip-logreg/models.npz", allow_pickle=True) as f:
    models = {k: f[k].item() for k in ECHONEXT_TARGETS}

k = 5
n_prototypes = pip_embeddings.shape[1]
n_labels = labels.shape[1]
prot_labels = np.zeros((n_prototypes, n_labels))
for i, label_name in enumerate(ECHONEXT_TARGETS):
    idxs = np.exp(models[label_name].coef_.squeeze()).argsort()[-k:]
    prot_labels[idxs, i] = 1
axs = scatter(pip_prototypes, ECHONEXT_TARGETS, prot_labels)

In [ ]:
# this approach computes weighted label assignments
probs = softmax(torch.as_tensor(pip_embeddings), dim=1).numpy() # (N, P) - for a given sample, what is probability it belongs to each prototype (each sample's probs sum to 1)
prot_label_scores = probs.T @ labels # (P, L) use the probs to compute a weighted label assignment for each prototype
prot_labels = prot_label_scores / probs.sum(axis=0)[:, None] # normalize by the sum of weights within each prototype

axs = scatter(pip_prototypes, ECHONEXT_TARGETS, prot_labels)
ax = axs[-1, -1]
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(pip_prototypes)
ax.scatter(coords[:, 0], coords[:, 1], c=probs.mean(axis=0)) # compare to overall similarity of training samples to each prototype

In [ ]:
# try k nearest neighbors majority vote approach (k nearest samples to each prototype)
k = 50
# threshold = k // 2
threshold = 5
knn_idxs = pip_embeddings.argsort(axis=0)[-k:, :]
print(f"Number of unique samples for top-{k}:", len(set(knn_idxs.flatten()))) # number of unique samples (max is k * n_prototypes)
prot_labels = (labels[knn_idxs].sum(axis=0) > threshold).astype(int) # majority vote
axs = scatter(pip_prototypes, ECHONEXT_TARGETS, prot_labels)

In [ ]:
# try k nearest prototypes for samples belonging to each label
k = 10
n_prototypes = pip_embeddings.shape[1]
n_labels = labels.shape[1]
prot_labels = np.zeros((n_prototypes, n_labels))
for i in range(n_labels):
    mask = labels[:, i] == 1
    prot_idxs = pip_embeddings[mask].mean(axis=0).argsort()[-k:]
    prot_labels[prot_idxs, i] = 1
axs = scatter(pip_prototypes, ECHONEXT_TARGETS, prot_labels)

In [ ]:
# try k nearest prototypes for samples belonging to each label
# additionally, only do a PCA over the prototypes that "matter"
k = 10
n_prototypes = pip_embeddings.shape[1]
n_labels = labels.shape[1]
prot_labels = np.zeros((n_prototypes, n_labels))
for i in range(n_labels):
    mask = labels[:, i] == 1
    prot_idxs = pip_embeddings[mask].mean(axis=0).argsort()[-k:]
    prot_labels[prot_idxs, i] = 1
mask = (prot_labels == 1).any(axis=1)
temp_prototypes = pip_prototypes[mask]
prot_labels = prot_labels[mask]
axs = scatter(temp_prototypes, ECHONEXT_TARGETS, prot_labels)

### EchoNext PIT (at varying scales)

In [ ]:
split_suffixes = ["", "-32k", "-16k", "-8k", "-4k", "-2k", "-1k", "-512", "-256"]

In [ ]:
suffix = ""
pit_meta_path = f"/opt/gpudata/ecg/echonext{suffix}/EchoNext_metadata_100k.csv"
pit_ckpt_path = f"../outputs/runs{suffix}/pass-heedb-pit/project-prototypes/latest"
pit_embs_path = f"../outputs/runs{suffix}/pass-heedb-pit-logreg/compute-embeddings/latest"

In [ ]:
labels, label_meta = get_echonext_labels(pit_meta_path)
pit_prototypes, pit_prot_meta = get_prototypes(pit_ckpt_path)

In [ ]:
label_meta["idx"] = np.arange(len(labels))
idxs = label_meta.set_index("ecg_key").loc[pit_prot_meta["ecg_id"], "idx"].to_numpy()
prot_labels = labels[idxs]

In [ ]:
scatter(pit_prototypes, ECHONEXT_TARGETS, prot_labels)